# 🌊 VaayuChakshu — Flood Monitoring (No-ZIP Colab Version)
**Instructions:** Just click **Run All** at the top!

This notebook will automatically:
1. Recreate your entire project codebase on the Colab server.
2. Download the satellite dataset.
3. Install dependencies.
4. Train the model on the T4 GPU.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('scripts', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
print('Directories created on Colab.')

In [ ]:
%%writefile config.yaml
# Project and Logging
project_name: "Flood Monitoring"
run_name: "resnet34-physics-loss-v2"
manifest_path: "data/processed/data_manifest.csv"

# Model Hyperparameters
learning_rate: 0.0002
adam_beta1: 0.5
adam_beta2: 0.999
ema_decay: 0.999

# Scheduler Settings
scheduler_factor: 0.5
scheduler_patience: 10

# Discriminator Update Frequency
discriminator_update_freq: 1

# Loss Weights (Lambdas)
lambda_l1: 100.0
lambda_adv: 1.0
lambda_perc: 10.0
lambda_speckle: 1.0
lambda_water: 5.0

# Training Settings
max_epochs: 200
batch_size: 8
accelerator: "auto"
precision: "16-mixed"
log_every_n_steps: 25

# Dataset and DataLoader Settings
num_workers: 0
cloud_threshold: 10.0
use_speckle_filter: True
pin_memory: True
persistent_workers: False



In [ ]:
%%writefile src/train.py
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(__file__)))
import yaml
os.environ["WANDB_MODE"] = "offline"
from argparse import ArgumentParser, Namespace
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping
from lightning.pytorch.loggers import WandbLogger
import logging
from logging_utils import setup_logging

from lightning_module import SAR2OpticalGAN
from datamodule import SARDataModule

setup_logging()
logger = logging.getLogger(__name__)

def main(hparams):
    logger.info("Initialising W&B logger for project '%s' run '%s'", hparams.project_name, hparams.run_name)
    wandb_logger = WandbLogger(
        project=hparams.project_name,
        name=hparams.run_name,
    )

    logger.debug("Logging hyperparameters to W&B")
    wandb_logger.log_hyperparams(hparams)

    logger.info("Creating model checkpoint callback")
    checkpoint_callback = ModelCheckpoint(
        dirpath=f"checkpoints/{hparams.run_name}",
        filename='{epoch:02d}-{val_psnr:.4f}',
        save_top_k=3,
        verbose=True,
        monitor='val/psnr',
        every_n_epochs=1,
        mode='max',
        save_last=True
    )
    early_stopping = EarlyStopping(monitor='train_loss/generator_total', mode='min', patience=10, verbose=True)

    lr_monitor = LearningRateMonitor(logging_interval='step')

    logger.info("Instantiating GAN model and datamodule")
    model = SAR2OpticalGAN(hparams)
    datamodule = SARDataModule(hparams)

    logger.info("Initialising Trainer with max_epochs=%s", hparams.max_epochs)
    trainer = Trainer(
        max_epochs=hparams.max_epochs,
        logger=wandb_logger,
        check_val_every_n_epoch=5,
        callbacks=[checkpoint_callback, early_stopping, lr_monitor],
        accelerator=hparams.accelerator,
        devices=1,
        strategy="auto",
        precision=hparams.precision,
        log_every_n_steps=hparams.log_every_n_steps,
    )
    logger.info("Starting training...")
    trainer.fit(model, datamodule)
    logger.info("Training complete")

    logger.debug("Closing W&B experiment")
    wandb_logger.experiment.finish()

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--config', type=str, required=True, help='Path to the YAML configuration file.')
    args = parser.parse_args()

    logger.debug("Loading YAML configuration from %s", args.config)
    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    hparams = Namespace(**config)
    logger.debug("Parsed hyperparameters: %s", hparams)
    try:
        main(hparams)
    except Exception as e:
        logger.exception("Unhandled exception during training: %s", str(e))
        raise


In [ ]:
%%writefile src/model.py
import torch
import torch.nn as nn
from torchvision.models import resnet34, ResNet34_Weights
import logging
from logging_utils import setup_logging
import torch.nn.functional as F
setup_logging()
logger = logging.getLogger(__name__)

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

class AttentionBlock(nn.Module):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x_flat = x.view(B, C, -1).permute(0, 2, 1)

        x_norm = self.norm1(x_flat)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x_flat + attn_out

        x_norm2 = self.norm2(x)
        mlp_out = self.mlp(x_norm2)
        x = x + mlp_out

        x = x.permute(0, 2, 1).view(B, C, H, W)
        return x

class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=4, base=64):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base*2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base*2, base*4)
        self.pool3 = nn.MaxPool2d(2)
        
        self.bottleneck = ConvBlock(base*4, base*8)
        self.attn = AttentionBlock(dim=base*8)

        self.up3 = UpBlock(base*8, base*4)
        self.up2 = UpBlock(base*4, base*2)
        self.up1 = UpBlock(base*2, base)

        self.out_conv = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool1(x1))
        x3 = self.enc3(self.pool2(x2))

        x4 = self.bottleneck(self.pool3(x3))
        x4 = self.attn(x4)

        x = self.up3(x4, x3)
        x = self.up2(x, x2)
        x = self.up1(x, x1)
        
        out = self.out_conv(x)
        return torch.tanh(out)

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=7):
        super().__init__()

        def discriminator_block(in_filters, out_filters, bn=True):
            return nn.Sequential(
                nn.Conv2d(in_filters, out_filters, 4, 2, 1, bias=False),
                nn.InstanceNorm2d(out_filters) if bn else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True)
            )

        self.model = nn.Sequential(
            discriminator_block(in_channels, 64, bn=False),
            discriminator_block(64, 128),
            discriminator_block(128, 256),
            discriminator_block(256, 512),
            discriminator_block(512, 1024),
            nn.Conv2d(1024, 1, kernel_size=4, padding=1)
        )

    def forward(self, x):
        return self.model(x)



In [ ]:
%%writefile src/dataset.py
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from data_utils import (load_and_stack_sar, load_and_stack_optical, load_cloud_mask, load_mask,
    apply_speckle_filter, normalize_sar, normalize_optical, get_cloud_coverage 
)
import albumentations as A
from albumentations.pytorch import ToTensorV2
import logging
from logging_utils import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

class FloodDataset(Dataset):
    def __init__(self, manifest_path, split='train', augment=True):
        """
        Custom dataset for flood mapping using Sentinel-2 optical and Sentinel-1 SAR images.
        Args: 
            manifest_path (str): Path to the dataset manifest CSV file.
            split (str): Dataset split - 'train', 'val', or 'test'.
            augment (bool): Whether to apply data augmentations.
        """
        self.df = pd.read_csv(manifest_path)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.split = split
        self.augment = augment

        logger.info(f"[{split}] Total samples: {len(self.df)}")

        additional = {'image0': 'image', 'mask0': 'mask', 'mask1': 'mask'}

        if self.augment and split == 'train':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                ToTensorV2(),
            ], additional_targets=additional)
        else:
            self.transform = A.Compose([ToTensorV2()], additional_targets=additional)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
    
        try:
            # Load SAR image (3 channels: VV, VH, VV-VH)
            sar_img = load_and_stack_sar(row['s1_vv'], row['s1_vh'])  # Lee filtering done here
            sar_img = normalize_sar(sar_img)
    
            # Load RGB optical image (3 channels)
            opt_img = load_and_stack_optical(
                row['s2_b4_red'], row['s2_b3_green'], row['s2_b2_blue'])
            opt_img = normalize_optical(opt_img)
    
            # Load masks
            cloud_mask = load_mask(row['s2_cloudmask'])
            water_mask = load_mask(row['s1_vv'].replace('VV.tif', 'LabelWater.tif'))
    
            # Move to HWC for albumentations
            sar_img = np.moveaxis(sar_img, 0, -1)
            opt_img = np.moveaxis(opt_img, 0, -1)
    
            # Apply augmentations
            augmented = self.transform(
                image=sar_img,
                image0=opt_img,
                mask0=cloud_mask,
                mask1=water_mask
            )
    
            # Convert augmented outputs to tensors (assuming CHW from ToTensorV2)
            sar_tensor = augmented['image'].float()  # (3, H, W)
            opt_rgb = augmented['image0'].float()    # (3, H, W)
            water_mask = augmented['mask1'].float().unsqueeze(0)  # (1, H, W)
    
            # Ensure matching spatial dimensions and create 4-channel optical tensor
            if opt_rgb.size()[1:] != water_mask.size()[1:]:  # Check H, W dimensions
                raise ValueError(f"Spatial dimensions mismatch: opt_rgb {opt_rgb.size()}, water_mask {water_mask.size()}")
            opt_tensor = torch.cat((opt_rgb, water_mask), dim=0)  # (4, H, W)
    
            cloud_tensor = augmented['mask0'].float().unsqueeze(0)  # (1, H, W)
            water_tensor = water_mask  # (1, H, W)
    
            return sar_tensor, opt_tensor, cloud_tensor, water_tensor
    
        except Exception as e:
            logger.error(f"Error loading sample {idx}: {e}")
            return None





In [ ]:
%%writefile src/datamodule.py
from torch.utils.data._utils.collate import default_collate
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from dataset import FloodDataset
import logging
from logging_utils import setup_logging
import torch

setup_logging()
logger = logging.getLogger(__name__)

def collate_fn(batch):
    '''
    Custom collate function to filter out None values.
    Returns zero tensors if batch is empty to avoid training crashes.
    '''
    filtered = list(filter(lambda x: x is not None, batch))
    if len(filtered) != len(batch):
        logger.warning("Batch contained %s invalid samples; they were filtered out", len(batch) - len(filtered))
    if not filtered:
        h, w = 512, 512
        return (torch.zeros(4, 3, h, w),)
    return default_collate(filtered)

class SARDataModule(pl.LightningDataModule):
    def __init__(self, hparams):
        logger.info("Creating SARDataModule")
        super().__init__()
        self.save_hyperparameters(hparams)

    def setup(self, stage=None):
        logger.debug("Setting up datasets for stage=%s", stage)
        '''
        Called on every GPU/TPU in distributed training
        '''
        if stage == 'fit' or stage is None:
            logger.info("Instantiating FloodDataset for training and validation")
            self.train_dataset = FloodDataset(
                manifest_path=self.hparams.manifest_path,
                split='train',
                augment=True,
            )
            self.val_dataset = FloodDataset(
                manifest_path=self.hparams.manifest_path,
                split='val',
                augment=False,
            )

    def train_dataloader(self):
        logger.debug("Creating training DataLoader with batch_size=%s", self.hparams.batch_size)
        return DataLoader(
            self.train_dataset,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=self.hparams.get('num_workers', 4),
            pin_memory=self.hparams.get('pin_memory', True),
            persistent_workers=self.hparams.get('persistent_workers', True),
            collate_fn=collate_fn,
        )

    def val_dataloader(self):
        logger.debug("Creating validation DataLoader with batch_size=%s", self.hparams.batch_size)
        return DataLoader(
            self.val_dataset,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.get('num_workers', 4),
            pin_memory=self.hparams.get('pin_memory', True),
            persistent_workers=self.hparams.get('persistent_workers', True),
            collate_fn=collate_fn,
        )


In [ ]:
%%writefile src/data_utils.py
import numpy as np
import rasterio
from scipy.ndimage.filters import uniform_filter
from scipy.ndimage.measurements import variance
import logging
from logging_utils import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

def lee_filter(img, size):
    logger.debug("Applying Lee filter with window size=%s", size)
    img_mean = uniform_filter(img, (size, size))
    img_sqr_mean = uniform_filter(img ** 2, (size, size))
    img_variance = img_sqr_mean - img_mean ** 2
    overall_variance = variance(img)
    img_weights = img_variance / (img_variance + overall_variance + 1e-8)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output

def load_image(path):
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

def load_and_stack_sar(vv_path, vh_path):
    logger.debug("Loading SAR VV from %s and VH from %s", vv_path, vh_path)
    s1_vv = lee_filter(load_image(vv_path), size=5)
    s1_vh = lee_filter(load_image(vh_path), size=5)
    s1_diff = s1_vv - s1_vh
    return np.stack([s1_vv, s1_vh, s1_diff], axis=0)  # (3, H, W)

def load_and_stack_optical(r_path, g_path, b_path):
    logger.debug("Loading optical R=%s G=%s B=%s", r_path, g_path, b_path)
    s2_r = load_image(r_path)
    s2_g = load_image(g_path)
    s2_b = load_image(b_path)
    return np.stack([s2_r, s2_g, s2_b], axis=0)

def normalize_sar(sar_img):
    logger.debug("Normalizing SAR image to [-1, 1] range")
    norm_channels = []
    for i in range(sar_img.shape[0]):
        ch = sar_img[i]
        ch_min = ch.min()
        ch_max = ch.max()
        norm = 2 * (ch - ch_min) / (ch_max - ch_min + 1e-8) - 1
        norm_channels.append(norm)
    return np.stack(norm_channels, axis=0)

def normalize_optical(optical_img):
    logger.debug("Normalizing optical image to [-1, 1] collectively")
    min_val = optical_img.min()
    max_val = optical_img.max()
    return 2 * (optical_img - min_val) / (max_val - min_val + 1e-8) - 1

def load_mask(mask_path):
    with rasterio.open(mask_path) as src:
        mask = src.read(1).astype(np.uint8)
    return (mask > 0).astype(np.float32)

def load_cloud_mask(mask_path):
    """Alias for load_mask — loads a binary cloud mask from a GeoTIFF."""
    return load_mask(mask_path)

def apply_speckle_filter(img, size=5):
    """Apply a Lee speckle filter to a single-channel SAR image."""
    logger.debug("Applying speckle (Lee) filter with window size=%s", size)
    return lee_filter(img, size=size)

def get_cloud_coverage(cloud_mask):
    """Return the fraction of pixels flagged as cloud (0.0 – 1.0)."""
    return float(cloud_mask.mean())


In [ ]:
%%writefile src/lightning_module.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import wandb
from argparse import Namespace
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from torch.optim.swa_utils import AveragedModel
from copy import deepcopy
import torch.distributed as dist

from model import UNetGenerator, PatchGANDiscriminator
from losses import LSGANLoss, PerceptualLoss, SpecklePreservationLoss, WaterIndexConsistencyLoss

class SAR2OpticalGAN(pl.LightningModule):
    def __init__(self, hparams: Namespace):
        super().__init__()
        self.save_hyperparameters(hparams)
        self.automatic_optimization = False

        self.generator = UNetGenerator(in_channels=3, out_channels=4)
        self.discriminator = PatchGANDiscriminator(in_channels=7)

        self.generator_ema = AveragedModel(
            deepcopy(self.generator),
            avg_fn=torch.optim.swa_utils.get_ema_avg_fn(self.hparams.ema_decay)
        )

        self.adv_loss = LSGANLoss(target_real_label=0.9, target_fake_label=0.05)
        self.l1_loss = nn.L1Loss()
        self.perceptual_loss = PerceptualLoss()
        self.speckle_loss = SpecklePreservationLoss()
        self.water_loss = WaterIndexConsistencyLoss()
        self.val_psnr = PeakSignalNoiseRatio(data_range=1.0)
        self.val_ssim = StructuralSimilarityIndexMeasure(data_range=1.0)

    def on_train_start(self):
        self.generator_ema.to(self.device)

    def forward(self, sar_img):
        return self.generator_ema(sar_img)

    def training_step(self, batch, batch_idx):
        opt_g, opt_d = self.optimizers()
        sar_img, opt_img, cloud_mask, water_mask = batch
        clear_pixels_mask = 1 - cloud_mask

        if self.trainer.world_size > 1:
            dist.barrier()

        try:
            with torch.cuda.amp.autocast(enabled=False):
                generated_opt = self.generator(sar_img)

                if torch.isnan(generated_opt).any():
                    raise ValueError("NaN in generated output")

                pred_fake_for_g = self.discriminator(torch.cat((sar_img, generated_opt), dim=1))

                loss_g_adv = self.hparams.lambda_adv * self.adv_loss(pred_fake_for_g, True)
                masked_fake_rgb = generated_opt[:, :3, :, :] * clear_pixels_mask
                masked_real_rgb = opt_img[:, :3, :, :] * clear_pixels_mask

                loss_g_l1 = self.hparams.lambda_l1 * self.l1_loss(masked_fake_rgb, masked_real_rgb)
                loss_g_perc = self.hparams.lambda_perc * self.perceptual_loss(masked_fake_rgb, masked_real_rgb, cloud_mask)
                loss_g_speckle = self.hparams.lambda_speckle * self.speckle_loss(generated_opt[:, :3, :, :], sar_img, cloud_mask)
                loss_g_water = self.hparams.lambda_water * self.water_loss(generated_opt, water_mask)

                loss_g = loss_g_adv + loss_g_l1 + loss_g_perc + loss_g_speckle + loss_g_water

                if torch.isnan(loss_g):
                    raise ValueError("NaN in generator total loss")

            opt_g.zero_grad()
            self.manual_backward(loss_g)
            torch.nn.utils.clip_grad_norm_(self.generator.parameters(), max_norm=1.0)
            opt_g.step()

            self.generator_ema.update_parameters(self.generator)

            loss_d = None
            if (batch_idx + 1) % self.hparams.discriminator_update_freq == 0:
                with torch.cuda.amp.autocast(enabled=False):
                    generated_opt_detached = generated_opt.detach()
                    pred_real = self.discriminator(torch.cat((sar_img, opt_img), dim=1))
                    loss_d_real = self.adv_loss(pred_real, True)
                    pred_fake = self.discriminator(torch.cat((sar_img, generated_opt_detached), dim=1))
                    loss_d_fake = self.adv_loss(pred_fake, False)
                    loss_d = (loss_d_real + loss_d_fake) * 0.5

                    if torch.isnan(loss_d):
                        raise ValueError("NaN in discriminator loss")

                opt_d.zero_grad()
                self.manual_backward(loss_d)
                torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), max_norm=1.0)
                opt_d.step()

            if self.trainer.is_global_zero:
                self.log("train/g_loss", loss_g, prog_bar=True, logger=False)
                log_dict = {
                    'train_loss/generator_total': loss_g,
                    'train_loss/g_adversarial': loss_g_adv,
                    'train_loss/g_l1': loss_g_l1,
                    'train_loss/g_perceptual': loss_g_perc,
                    'train_loss/g_speckle': loss_g_speckle,
                    'train_loss/g_water_consistency': loss_g_water
                }
                if loss_d is not None:
                    log_dict['train/d_loss'] = loss_d
                    log_dict['train_loss/discriminator'] = loss_d
                self.log_dict(log_dict, logger=True)

            if self.current_epoch % 5 == 0 and batch_idx == 0 and self.logger and self.logger.experiment:
                num_samples = min(10, sar_img.size(0))
                images = []
                for i in range(num_samples):
                    sar = sar_img[i]
                    opt = opt_img[i, :3, :, :]
                    cloud = cloud_mask[i]
                    water = water_mask[i]
                    gen = generated_opt[i]
    
                    sar_vv = sar[0].unsqueeze(0).repeat(3, 1, 1)
                    cloud_3c = cloud.repeat(3, 1, 1)
                    water_3c = water.repeat(3, 1, 1)
                    gen_rgb = gen[:3]
    
                    image = torch.cat([sar_vv, opt, cloud_3c, water_3c, gen_rgb], dim=2)
                    images.append(wandb.Image(image, caption=f"Sample {i} | SAR | OPT | Cloud | Water | Gen | Epoch {self.current_epoch}"))
    
                self.logger.experiment.log({
                    "train_samples": images,
                    "epoch": self.current_epoch
                })
                
                # Checkpoint syncing is handled by ModelCheckpoint callback (local run)
            
        
        except Exception as e:
            self.logger.experiment.log({"error": str(e), "epoch": self.current_epoch})
            raise e

        return loss_g

    def validation_step(self, batch, batch_idx):
        sar_img, opt_img, cloud_mask, water_mask = batch
        generated_opt = self(sar_img)

        val_l1_loss = self.l1_loss(generated_opt[:, :3, :, :], opt_img[:, :3, :, :])

        opt_img_01 = (opt_img[:, :3, :, :] + 1) / 2
        generated_opt_01 = (generated_opt[:, :3, :, :] + 1) / 2

        self.val_psnr(generated_opt_01, opt_img_01)
        self.val_ssim(generated_opt_01, opt_img_01)

        self.log_dict(
            {'val/psnr': self.val_psnr, 'val/ssim': self.val_ssim, 'val/l1_loss': val_l1_loss},
            on_epoch=True, prog_bar=True, sync_dist=True
        )
        if batch_idx == 0 and self.trainer.is_global_zero and self.logger and self.logger.experiment:
            num_samples = min(10, sar_img.size(0))
            images = []
            for i in range(num_samples):
                sar = sar_img[i]           # (3, 512, 512)
                opt = opt_img[i, :3, :, :] # Use only RGB channels (3, 512, 512)
                cloud = cloud_mask[i]      # (1, 512, 512)
                water = water_mask[i]      # (1, 512, 512)
                gen = generated_opt[i]     # (4, 512, 512)
    
                sar_vv = sar[0].unsqueeze(0).repeat(3, 1, 1)
                cloud_3c = cloud.repeat(3, 1, 1)
                water_3c = water.repeat(3, 1, 1)
                gen_rgb = gen[:3]
                image = torch.cat([sar_vv, opt, cloud_3c, water_3c, gen_rgb], dim=2)
                images.append(wandb.Image(image, caption=f"Sample {i} | SAR | OPT | Cloud | Water | Gen"))
    
            self.logger.experiment.log({
                "val_samples": images,
                "epoch": self.current_epoch
            })

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        sch_g, sch_d = self.lr_schedulers()
        psnr_metric = self.trainer.callback_metrics.get("val/psnr")
        d_loss_metric = self.trainer.callback_metrics.get("train/d_loss_epoch")
        if psnr_metric is not None:
            sch_g.step(psnr_metric)
        if d_loss_metric is not None:
            sch_d.step(d_loss_metric)

    def configure_optimizers(self):
        opt_g = torch.optim.AdamW(self.generator.parameters(), lr=1e-5, betas=(0.5, 0.999))
        opt_d = torch.optim.AdamW(self.discriminator.parameters(), lr=1e-5, betas=(0.5, 0.999))

        scheduler_g = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt_g, mode='max', factor=self.hparams.scheduler_factor,
            patience=self.hparams.scheduler_patience
        )
        scheduler_d = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt_d, mode='min', factor=self.hparams.scheduler_factor,
            patience=self.hparams.scheduler_patience
        )

        return (
            {"optimizer": opt_g, "lr_scheduler": {"scheduler": scheduler_g, "monitor": "val/psnr"}},
            {"optimizer": opt_d, "lr_scheduler": {"scheduler": scheduler_d, "monitor": "train/d_loss_epoch"}}
        )

    def on_train_batch_start(self, batch, batch_idx):
        batch = [t.to(self.device) for t in batch]



In [ ]:
%%writefile src/losses.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg19, VGG19_Weights
import logging
from logging_utils import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

#1. Adversarial Loss
class LSGANLoss(nn.Module):
    def __init__(self, target_real_label=0.9, target_fake_label=0.0):
        '''
        Adversarial loss for the generator.
        Args:
            target_real_label (float): Target label for real images.
            target_fake_label (float): Target label for fake images.
        '''
        super(LSGANLoss, self).__init__()
        self.register_buffer('real_label', torch.tensor(target_real_label))
        self.register_buffer('fake_label', torch.tensor(target_fake_label))
        self.loss = nn.MSELoss()

    def forward(self, prediction, target_is_real):
        logger.debug("LSGANLoss called, target_is_real=%s", target_is_real)
        target_tensor = self.real_label if target_is_real else self.fake_label
        return self.loss(prediction, target_tensor.expand_as(prediction))

# 2. Perceptual Loss - VGG19 Feature Loss
class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:36].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.loss = nn.L1Loss()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, gen_img, real_img, cloud_mask):
        logger.debug("PerceptualLoss forward with gen_img shape %s", tuple(gen_img.shape))
        gen_img = (gen_img + 1) / 2
        real_img = (real_img + 1) / 2
        gen_img = (gen_img - self.mean) / self.std
        real_img = (real_img - self.mean) / self.std

        gen_features = self.vgg(gen_img)
        real_features = self.vgg(real_img)
        
        mask = F.interpolate(cloud_mask, size=gen_features.shape[2:], mode='bilinear', align_corners=False)
        mask = (1 - mask).expand_as(gen_features)
        masked_gen_features = gen_features * mask
        masked_real_features = real_features * mask
        return self.loss(masked_gen_features, masked_real_features)

# 3. Custom Speckle Preservation Loss
class SpecklePreservationLoss(nn.Module):
    def __init__(self):
        super(SpecklePreservationLoss, self).__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)
        self.loss = nn.L1Loss()

    def get_gradient_magnitude(self, img):
        if img.shape[1] == 3:
            img_gray = 0.299 * img[:, 0:1, :, :] + 0.587 * img[:, 1:2, :, :] + 0.114 * img[:, 2:3, :, :]
        elif img.shape[1] == 2:
            img_gray = img[:, 0:1, :, :]
        else:
            img_gray = img
        grad_x = F.conv2d(img_gray, self.sobel_x, padding='same')
        grad_y = F.conv2d(img_gray, self.sobel_y, padding='same')
        return torch.sqrt(grad_x**2 + grad_y**2 + 1e-6)

    def forward(self, gen_img, sar_img, cloud_mask):
        logger.debug("SpecklePreservationLoss forward")
        with torch.cuda.amp.autocast():
            grad_gen = self.get_gradient_magnitude(gen_img)
            grad_sar = self.get_gradient_magnitude(sar_img)
        weight_map = torch.exp(-grad_sar)
        
        mask = (1 - cloud_mask).expand_as(grad_gen)
        weighted_grad_gen = weight_map * grad_gen * mask
        return self.loss(weighted_grad_gen, torch.zeros_like(weighted_grad_gen))

# 4. WaterIndexConsistencyLoss
class WaterIndexConsistencyLoss(nn.Module):
    def __init__(self, ndwi_weight=0.5):
        super(WaterIndexConsistencyLoss, self).__init__()
        self.bce_loss = nn.BCEWithLogitsLoss()
        # ndwi_weight is kept for backward compatibility but won't be used
        self.ndwi_weight = ndwi_weight

    def forward(self, gen_img, water_mask):
        logger.debug("WaterIndexConsistencyLoss forward with gen_img shape %s", gen_img.shape)

        # The 4th channel is the water mask logits
        water_logits = gen_img[:, 3:4, :, :]
        
        # We drop the NDWI proxy loss because calculating (G-R)/(G+R) was causing
        # mode collapse (forcing the model to output purely Red and Green).
        total_loss = self.bce_loss(water_logits, water_mask)
        return total_loss


In [ ]:
%%writefile src/logging_utils.py
import logging
import os
from datetime import datetime


def setup_logging(log_level: int = logging.INFO, log_dir: str | None = None, log_file: str | None = None):
    """Configure root logger with console and optional file handlers.

    Parameters
    ----------
    log_level: int
        Logging level for root logger. Defaults to ``logging.INFO``.
    log_dir: str | None
        Folder where log file should be saved. Created if it does not exist.
        If *None*, a ``logs`` directory is created in the project root.
    log_file: str | None
        Log filename. If *None*, a timestamped filename is generated.
    """

    # Ensure we only configure logging once in interactive / multi-process settings.
    if getattr(setup_logging, "_is_configured", False):  # type: ignore[attr-defined]
        return

    logger = logging.getLogger()
    logger.setLevel(log_level)

    formatter = logging.Formatter(
        fmt="%(asctime)s — %(levelname)s — %(name)s — %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(log_level)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    # Optional file handler
    if log_dir is None:
        log_dir = os.path.join(os.getcwd(), "logs")
    if not os.path.exists(log_dir):
        os.makedirs(log_dir, exist_ok=True)

    if log_file is None:
        log_file = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    file_path = os.path.join(log_dir, log_file)

    file_handler = logging.FileHandler(file_path)
    file_handler.setLevel(log_level)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    logger.debug("Logging configured. Logs will be written to %s", file_path)

    # Mark as configured
    setup_logging._is_configured = True  # type: ignore[attr-defined] 


In [ ]:
%%writefile scripts/fetch_data.py
import datetime as dt
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from sentinelhub import (
    CRS,  #coordinate reference system
    BBox,  #define a bounding box
    DataCollection,  #Specify the satellite dataset you want
    Geometry,  #can represet a point, line, or polygon
    MimeType,  #specify the format of the data you want
    SentinelHubCatalog,  #for searching and retrieving data
    SentinelHubRequest,  #for making requests to Sentinel Hub
    SHConfig,  #configuration for Sentinel Hub
    bbox_to_dimensions,  #convert a bounding box to dimensions
)
from tifffile import imwrite

load_dotenv()


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

CLIENT_ID = os.getenv('SH_CLIENT_ID')
CLIENT_SECRET = os.getenv('SH_CLIENT_SECRET')

config = SHConfig()

if CLIENT_ID and CLIENT_SECRET:
    config.sh_client_id = CLIENT_ID
    config.sh_client_secret = CLIENT_SECRET
    logging.info("Sentinel Hub configuration set with client ID and secret.")
else:
    logging.error("Sentinel Hub client ID and secret are not set. Please check your .env file.")
    raise ValueError("Sentinel Hub client ID and secret are not set.")

config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"


WORKDIR = os.environ.get('WORKDIR', str(Path.cwd()))
WORKDIR = Path(WORKDIR)

OUTPUT_DIR = WORKDIR / 'data' / 'raw'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logging.info(f"Output directory set to: {OUTPUT_DIR}")

AOI_BBOX = [6.9, 50.5, 7.3, 50.6]  # Ahr Valley, Germany
AOI_CRS = CRS.WGS84  # these values are in degrees, using WGS84
TIME_INTERVAL = ('2021-07-01', '2021-07-25')
RESOLUTION = 10  # in meters


evalscript_s1 = """
//VERSION=3
function setup() {
    return {
        input: ["VV", "VH"],
        output: { bands: 2, sampleType: "FLOAT32" }
    };
}
function evaluatePixel(sample) {
    return [sample.VV, sample.VH];
}
"""

evalscript_s2 = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04", "B08", "SCL", "dataMask"],
    output: { bands: 5, sampleType: "UINT16" }
  };
}
function evaluatePixel(sample) {
  if (sample.dataMask == 0) {
      return [0, 0, 0, 0, 0];
  }
  let cloudMask = 0;
  if (sample.SCL == 3 || sample.SCL == 8 || sample.SCL == 9 || sample.SCL == 10) {
      cloudMask = 1;
  }
  return [sample.B02, sample.B03, sample.B04, sample.B08, cloudMask];
}
"""

def search_available_scenes(catalog: SentinelHubCatalog, data_collection: DataCollection, geometry: Geometry, time_interval: tuple) -> list[str]:
    """
    Search for available scenes in the specified area and time interval.
    
    :param catalog: SentinelHubCatalog instance
    :param data_collection: DataCollection to search in
    :param geometry: Geometry of the area of interest
    :param time_interval: Tuple with start and end date
    :return: List of scene IDs
    """
    logging.info(f"Searching for scenes in {data_collection.name} from {time_interval[0]} to {time_interval[1]}")

    search_iterator = catalog.search(
        collection=data_collection,
        geometry=geometry,
        time=time_interval,
        fields={"include": ["properties.datetime"], "exclude": []},
    )
    timestamps = [item["properties"]["datetime"] for item in search_iterator]
    logging.info(f"Found {len(timestamps)} scenes.")
    return sorted(list(set(timestamps)))  # Return unique timestamps sorted

def download_scenes(timestamps: list, data_collection:DataCollection, evalscript: str, file_prefix: str):
    """
    Download scenes for the given timestamps from Sentinel Hub.
    :param timestamps: List of timestamps to download
    :param data_collection: DataCollection to download from
    :param evalscript: Evalscript for processing the data
    :param file_prefix: Prefix for the output files
    """
    if not timestamps:
        logging.warning("No timestamps provided for downloading scenes.")
        return

    logging.info(f"Downloading {len(timestamps)} scenes from {data_collection.name}... with prefix '{file_prefix}'")
    aoi_bbox_obj = BBox(AOI_BBOX, crs=AOI_CRS)
    size = bbox_to_dimensions(aoi_bbox_obj, resolution=RESOLUTION)

    MAX_SIZE = 2500 # Sentinel Hub limits the size of the request
    if size[0] > MAX_SIZE or size[1] > MAX_SIZE:
        scale = min(MAX_SIZE / size[0], MAX_SIZE / size[1])
        size = (int(size[0] * scale), int(size[1] * scale))
        logging.warning(f"Resized dimensions to fit API limits: {size}")

    for ts in timestamps:
        acquisition_time = dt.datetime.fromisoformat(ts.replace('Z', '+00:00')) # Convert to datetime object
        time_slot = (
            acquisition_time - dt.timedelta(minutes=30),
            acquisition_time + dt.timedelta(minutes=30)
        )
        logging.debug(f"Processing scene for time slot: {time_slot}")

        request = SentinelHubRequest(
            evalscript=evalscript,
            input_data=[
                SentinelHubRequest.input_data(
                    data_collection=data_collection,
                    time_interval=time_slot,
                )
            ],
            responses=[
                SentinelHubRequest.output_response(
                    "default", MimeType.TIFF                )
            ],
            bbox=aoi_bbox_obj,
            size=size,
            config=config,
        )

        try:
            data = request.get_data()[0]
            filename_ts = acquisition_time.strftime("%Y%m%dT%H%M%S")
            filename = OUTPUT_DIR / f"{file_prefix}_{filename_ts}.tiff"
            imwrite(filename, data)
            logging.info(f"Downloaded scene for {acquisition_time} to {filename}")
        except Exception as e:
            logging.error(f"Failed to download scene for {acquisition_time}: {e}")


if __name__ == "__main__":
    aoi_geometry = Geometry(BBox(AOI_BBOX, crs=AOI_CRS).geometry, crs=AOI_CRS)
    catalog = SentinelHubCatalog(config=config)
    logging.info("Starting data fetch process...")

    # Search and download Sentinel-1 scenes
    S1_COLLECTION = DataCollection.define(
        name = "SENTINEL1_GRD_CDSE",
        api_id = "sentinel-1-grd"
    )
    S2_COLLECTION = DataCollection.define(
        name = "SENTINEL2_L2A_CDSE",
        api_id = "sentinel-2-l2a"
    )
    #S1 - Sentinel-1 GRD (Ground Range Detected) SAR radar data
    s1_timestamps = search_available_scenes(catalog, S1_COLLECTION, aoi_geometry, TIME_INTERVAL)
    #S2 - Sentinel-2 L2A (Level 2A) optical data with cloud masking
    s2_timestamps = search_available_scenes(catalog, S2_COLLECTION, aoi_geometry, TIME_INTERVAL)

    logging.info("-" * 50)
    if s1_timestamps: logging.info(f"Found {len(s1_timestamps)} available Sentinel-1 scenes.")
    else: logging.warning("No Sentinel-1 data found.")
    if s2_timestamps: logging.info(f"Found {len(s2_timestamps)} available Sentinel-2 scenes.")
    else: logging.warning("No Sentinel-2 data found.")
    logging.info("-" * 50)

    download_scenes(s1_timestamps, S1_COLLECTION, evalscript_s1, "s1_iw_grd")
    download_scenes(s2_timestamps, S2_COLLECTION, evalscript_s2, "s2_l2a")

    logging.info("\nData acquisition process complete.")



In [ ]:
%%writefile scripts/prepare_dataset.py
import os
import csv
import random
import logging
import re
from glob import glob

from src.logging_utils import setup_logging

setup_logging()
logger = logging.getLogger(__name__)


def create_dataset_manifest(data_root, output_csv, test_size=0.15, random_state=42):
    """
    Parses the C2S-MS Floods dataset structure, correctly pairs S1 and S2 chips based on the
    actual file naming convention, and creates a manifest file with train/val splits.
    Only uses Python standard libraries to avoid external dependency requirements.

    Args:
        data_root (str): Root directory where the event folders (UUIDs) are located.
        output_csv (str): Path to save the output CSV manifest file.
        test_size (float): The proportion of the dataset to allocate to the validation set.
        random_state (int): Seed for the random split for reproducibility.
    """
    try:
        # Search for event directories
        event_dirs = [d for d in glob(os.path.join(data_root, '*')) if os.path.isdir(d)]
        if not event_dirs:
            logger.error(f"No event directories found in {data_root}. Please check the path.")
            return
        logger.info(f"Found {len(event_dirs)} event directories. Starting scan...")

        records = []
        # regex to extract the coordinate ID (e.g. - '01845-00514')
        id_parser = re.compile(r'(\d{5}-\d{5})$')
        
        for event_dir in event_dirs:
            event_name = os.path.basename(event_dir)
            s1_chip_dirs = glob(os.path.join(event_dir, 's1', '*'))
            s2_chip_dirs = glob(os.path.join(event_dir, 's2', '*'))

            # Create dictionaries mapping the unique ID to the full path
            s1_map = {}
            for p in s1_chip_dirs:
                match = id_parser.search(os.path.basename(p))
                if match:
                    s1_map[match.group(1)] = p
            
            s2_map = {}
            for p in s2_chip_dirs:
                match = id_parser.search(os.path.basename(p))
                if match:
                    s2_map[match.group(1)] = p

            common_ids = s1_map.keys() & s2_map.keys()

            if not common_ids:
                logger.warning(f"No matching chip IDs found for event {event_name}. Skipping event.")
                continue

            logger.info(f"Processing event {event_name}: Found {len(common_ids)} pairs.")

            for chip_id in common_ids:
                s1_chip_dir = s1_map[chip_id]
                s2_chip_dir = s2_map[chip_id]
                
                s1_vv_path = os.path.join(s1_chip_dir, 'VV.tif') 
                s1_vh_path = os.path.join(s1_chip_dir, 'VH.tif') 
                
                s2_b4_path = os.path.join(s2_chip_dir, 'B4.tif') 
                s2_b3_path = os.path.join(s2_chip_dir, 'B3.tif') 
                s2_b2_path = os.path.join(s2_chip_dir, 'B2.tif')
                
                s2_cloudmask_path = os.path.join(s2_chip_dir, 'LabelCloud.tif')

                required_files = [s1_vv_path, s1_vh_path, s2_b4_path, s2_b3_path, s2_b2_path, s2_cloudmask_path]
                if all(os.path.exists(p) for p in required_files):
                    records.append({
                        's1_vv': s1_vv_path.replace(os.sep, '/'),
                        's1_vh': s1_vh_path.replace(os.sep, '/'),
                        's2_b4_red': s2_b4_path.replace(os.sep, '/'),
                        's2_b3_green': s2_b3_path.replace(os.sep, '/'),
                        's2_b2_blue': s2_b2_path.replace(os.sep, '/'),
                        's2_cloudmask': s2_cloudmask_path.replace(os.sep, '/'),
                        'event': event_name,
                        's1_chip_id': os.path.basename(s1_chip_dir),
                        's2_chip_id': os.path.basename(s2_chip_dir)
                    })
                else:
                    logger.warning(f"Missing one or more files in chip pair: "
                                   f"S1: {os.path.basename(s1_chip_dir)}, S2: {os.path.basename(s2_chip_dir)}. Skipping.")

        if not records:
            logger.error("No valid records were created. This can happen if files are still missing or paths are incorrect.")
            return

        logger.info(f"Successfully processed {len(records)} complete chip pairs.")

        # Train/validation split using random (reproducible with seed)
        random.seed(random_state)
        random.shuffle(records)
        
        val_count = int(len(records) * test_size)
        
        for i, record in enumerate(records):
            if i < val_count:
                record['split'] = 'val'
            else:
                record['split'] = 'train'

        # Write to CSV
        output_dir = os.path.dirname(output_csv)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)

        headers = ['s1_vv', 's1_vh', 's2_b4_red', 's2_b3_green', 's2_b2_blue', 's2_cloudmask', 'event', 's1_chip_id', 's2_chip_id', 'split']
        with open(output_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            for record in records:
                writer.writerow(record)
                
        logger.info(f"Manifest file created successfully at: {output_csv}")
        
        train_count = len(records) - val_count
        logger.info(f"Dataset summary: train={train_count}, val={val_count}")

    except Exception as e:
        logger.exception(f"An unexpected error occurred: {e}")


if __name__ == '__main__':
    DATASET_ROOT = os.path.join("data", "raw", "data", "c2s_ms_floods", "chips")
    OUTPUT_MANIFEST = os.path.join("data", "processed", "data_manifest.csv")

    create_dataset_manifest(DATASET_ROOT, OUTPUT_MANIFEST)


In [ ]:
%%writefile inference.py
import os
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), 'src'))

import argparse
import torch
import numpy as np
import rasterio
from PIL import Image

from logging_utils import setup_logging
import logging

setup_logging()
logger = logging.getLogger(__name__)

from lightning_module import SAR2OpticalGAN
from data_utils import load_and_stack_sar, normalize_sar

def run_inference(checkpoint_path, s1_vv_path, s1_vh_path, output_dir="outputs/results"):
    """
    Runs inference on a Sentinel-1 SAR pair using a trained Flood-GAN checkpoint.
    Saves the generated optical RGB image and the flood water mask.
    """
    logger.info("Loading model from checkpoint: %s", checkpoint_path)
    # Load model (compiled weights)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SAR2OpticalGAN.load_from_checkpoint(checkpoint_path)
    model.to(device)
    model.eval()
    
    logger.info("Loading and preprocessing input SAR image...")
    # Load and stack VV and VH channels (applies Lee filter)
    sar_stacked = load_and_stack_sar(s1_vv_path, s1_vh_path)
    sar_normalized = normalize_sar(sar_stacked)
    
    # Add batch dimension and convert to torch tensor
    sar_tensor = torch.from_numpy(sar_normalized).float().unsqueeze(0).to(device) # (1, 3, H, W)
    
    logger.info("Running forward pass through Generator...")
    with torch.no_grad():
        # Generator outputs a 4-channel tensor (RGB + Water mask logits)
        generated_output = model(sar_tensor)
        
    # Remove batch dimension
    generated_output = generated_output.squeeze(0).cpu() # (4, H, W)
    
    # 1. Post-process Generated RGB Image
    rgb_output = generated_output[:3, :, :] # (3, H, W)
    # Convert from [-1, 1] scale back to [0, 255] uint8
    rgb_output = ((rgb_output + 1) / 2.0 * 255.0).clamp(0, 255).numpy().astype(np.uint8)
    rgb_image = np.moveaxis(rgb_output, 0, -1) # (H, W, 3)
    
    # 2. Post-process Generated Flood Water Mask
    water_logits = generated_output[3, :, :] # (H, W)
    # Apply sigmoid to get probabilities, then threshold at 0.5 for binary mask
    water_probs = torch.sigmoid(water_logits).numpy()
    water_mask = (water_probs > 0.5).astype(np.uint8) * 255 # (H, W)
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Save files
    s1_id = os.path.basename(os.path.dirname(s1_vv_path))
    rgb_save_path = os.path.join(output_dir, f"{s1_id}_gen_optical.png")
    mask_save_path = os.path.join(output_dir, f"{s1_id}_gen_flood_mask.png")
    
    Image.fromarray(rgb_image).save(rgb_save_path)
    Image.fromarray(water_mask).save(mask_save_path)
    
    logger.info("Inference complete!")
    logger.info("Saved generated optical image to: %s", rgb_save_path)
    logger.info("Saved generated flood mask to: %s", mask_save_path)

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Flood-GAN Inference Script")
    parser.add_argument("--checkpoint", type=str, required=True, help="Path to the trained PyTorch Lightning checkpoint file (.ckpt)")
    parser.add_argument("--s1_vv", type=str, required=True, help="Path to the input Sentinel-1 VV .tif file")
    parser.add_argument("--s1_vh", type=str, required=True, help="Path to the input Sentinel-1 VH .tif file")
    parser.add_argument("--output_dir", type=str, default="outputs/results", help="Directory to save the generated results")
    
    args = parser.parse_args()
    
    try:
        run_inference(args.checkpoint, args.s1_vv, args.s1_vh, args.output_dir)
    except Exception as e:
        logger.exception("Error during inference execution: %s", str(e))
        raise



## Install Dependencies

In [ ]:
!pip install -q pytorch-lightning==2.5.2 lightning==2.5.2 wandb==0.20.1 rasterio==1.4.3 albumentations==2.0.8 torchmetrics==1.7.3 onnx==1.18.0 python-dotenv==1.1.0 scikit-image==0.25.2 tifffile==2025.6.11 boto3
print('Dependencies installed!')

## Extract Dataset
**IMPORTANT**: Make sure you have uploaded `data.zip` to Colab before running this cell!

In [ ]:
import os
if not os.path.exists('data.zip'):
    raise FileNotFoundError('Please upload data.zip to Colab first!')
!unzip -q -o data.zip
print('Dataset extracted successfully!')

## Prepare Dataset Manifest
This generates the `data_manifest.csv` required for training.

In [ ]:
!python scripts/prepare_dataset.py

## 🚀 Start Training!

In [ ]:
import torch
import os
os.environ['WANDB_MODE'] = 'offline'
torch.set_float32_matmul_precision('medium')
!python -m src.train --config config.yaml